# Arc-length pipeline: model -> prompts -> activations -> arc lengths

Runs every stage for one model. All logic lives in `scripts/`; this notebook only
configures and orchestrates. Everything generated goes under `artifacts/<model name>/`:

| folder | contents |
|---|---|
| `datasets/` | generated prompt JSON, one file per corpus |
| `activations/` | final-token layer caches, one `.pt` per corpus/template |
| `surface/` | `rows.parquet` (with `arc_length_parallel`, `arc_length_orthogonal`), `model.npz`, `model.json`, mapping checkpoints |

Completed caches are reused after fingerprint checks, so rerunning skips inference.
Compatible stated caches placed in `activations/` are projected with frozen transforms.

In [ ]:
from pathlib import Path
import gc
import sys

import torch
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'stakes_surface_pipeline.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import cache_activations, prompt_datasets
from scripts.pipeline_config import NO_TIME_CORPORA, RunConfig
from scripts.stakes_surface_pipeline import StakesSurfacePipeline

print('Repository:', ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')

## Configuration

- `MODEL_NAME`: Hugging Face model id. Its name (without the org) becomes the artifacts subfolder.
  For gated models, run `huggingface-cli login` first.
- `BATCH_SIZE`: prompts per forward pass; reduce if GPU memory is insufficient.
- `STAKES_MERGES`: original stakes level -> merged class. Unlisted levels keep their own class.
  The seven BCPC directions need at least eight merged classes, and `very_low` must remain
  a class because it orients arc length.

Changing the layer or position for a model with existing caches raises a stale-cache error;
set `FORCE = True` to rebuild them.

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'
BATCH_SIZE = 256
STAKES_MERGES = {
    'near_existential': 'existential',
    'medium_low': 'medium',
}

LAYER_COMPONENT = 'layer_out/21'
POSITION = -1                    # final prompt token
CORPORA = list(NO_TIME_CORPORA)  # quick smoke run: ['conversational_no_time']
FORCE = False                    # rebuild caches that fail fingerprint checks

config = RunConfig(model_name=MODEL_NAME, layer_component=LAYER_COMPONENT, position=POSITION,
                   batch_size=BATCH_SIZE, stakes_merges=STAKES_MERGES)
print(config.describe())

## 1. Load the model
Loaded with left padding, an empty system prompt, and bfloat16 where supported.

In [ ]:
model, tokenizer = cache_activations.load_model(config)
print(f'Loaded {MODEL_NAME} on {model.device}')

## 2. Generate prompts
Writes one JSON per corpus to `datasets/`, then checks task counts, the prompt budget,
cross-register duplicates, and the stakes x duration crossing.

In [ ]:
datasets = prompt_datasets.generate_datasets(config, CORPORA)
prompt_datasets.preflight(datasets)

## 3. Cache activations
One cache per corpus/template preserves equal-file fitting weights. The model is released
afterwards; the remaining stages run on CPU.

In [ ]:
cache_paths = cache_activations.run(config, datasets, model=model, tokenizer=tokenizer, force=FORCE)
print(f'{len(cache_paths)} template caches in {config.activations_dir}')
del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Cache inventory
Checks homogeneous horizon type per file and consistent model, layer, position, and width.

In [ ]:
pipeline = StakesSurfacePipeline(config)
display(pipeline.load_caches())

## 5. BCPC target: `bcpc_arc_length`
Seven between-class directions of the merged stakes classes, weighted equally per file;
a penalized cubic spline through median class centers and two anchors; arc length starts
at the end nearest `very_low`.

In [ ]:
projected_rows = pipeline.fit_bcpc()
bcpc = pipeline.bcpc
print(f'{len(projected_rows):,} rows, {len(bcpc.projection["classes"])} merged classes')
display(bcpc.class_table())
display(bcpc.variance_table())
display(bcpc.centroids)
display(bcpc.anchors)
print(f'Weighted squared residual sum: {bcpc.weighted_residual_sum:.6g}')
display(bcpc.spline_points)
print(f's=0: spline endpoint associated with {bcpc.zero_anchor}')
print(f'Total spline arc length: {bcpc.total_arc_length:.6g}')
display(projected_rows[['stakes', 'spline_parameter', 'bcpc_arc_length', 'distance_to_spline']].head())

## 6. PLS: 16 components, equal weight per file
Centered and unscaled; each row weighs `1 / (files * rows_in_file)`.

In [ ]:
display(pipeline.fit_pls())
display(pipeline.weight_summary)

## 7. Centroid-plane rotation of PLS1-3
Per-file stakes-centroid splines; PLS1-3 rotate into the plane that favors between-file
separation along its normal. PLS4-16 keep their fitted axes.

In [ ]:
rotation_diagnostics, file_centroids = pipeline.rotate_plane()
display(rotation_diagnostics)
display(file_centroids)

## 8. Surface: PLS2 = f(PLS1, PLS3)
Equal-row cubic graph on horizon-free rows, with tangent extensions beyond the fitted PLS1 range.

In [ ]:
display(pipeline.fit_surface())

## 9. Project stated rows
Only runs on stated caches present in `activations/`; they never enter any fit.

In [ ]:
display(pipeline.project_stated_rows())

## 10. Slice cache and validation
5,000 fixed-PLS3 curves covering every row's height; a sample is checked against
independent minimization and quadrature, and the cell raises if accuracy fails.

In [ ]:
display(pipeline.build_slice_cache())

## 11. Arc lengths
`arc_length_parallel` is signed slice length from the common origin; `arc_length_orthogonal`
is snapped PLS3 height (legacy name). Mapping checkpoints allow resumption.

In [ ]:
mapping = pipeline.map_coordinates()
for table in mapping.values():
    display(table)
display(pipeline.all_rows[['source_file', 'stakes', 'bcpc_arc_length',
                           'arc_length_parallel', 'arc_length_orthogonal']].head())

## 12. Export

In [ ]:
output_dir = pipeline.export(notebook='notebooks/arc_length_pipeline.ipynb')

## 13. Diagnostic plots
Saved as self-contained HTML under `artifacts/<model name>/plots/`:
`bcpc_centroid_spline.html` (BCPC1-3 rows, class medians, anchors, spline),
`pls_centroid_splines.html` (rotated PLS1-3 rows, per-file class means and splines), and
`pls_surface.html` (height graph, slices, per-file splines). Sampling affects display only.

In [ ]:
figures = pipeline.save_plots()
for fig in figures.values():
    fig.show()

To project new activations without refitting:

```python
from scripts.stakes_surface_bundle import load_surface_bundle, project_saved_bcpc
arrays, metadata, saved_coordinates = load_surface_bundle(config.surface_dir)
bcpc_projection = project_saved_bcpc(activation_batch, arrays, metadata)
pls_scores = (activation_batch - arrays['pls_mean']) @ arrays['pls_rotations']
new_surface_coordinates = saved_coordinates.map_points(pls_scores[:, :3])
```

## 14. Inference-only severity prompts

The 20 editable templates in `scripts/corpora/severity_prompts.py` are cached separately and projected through the exported surface. They never enter fitting or slice-grid construction. Outputs live under `artifacts/<model name>/inference/severity/`. Compatible activation caches are reused; the model loads only when required. Coordinates use the current saved bundle on every run. `FORCE` also controls these caches.

The CSV contains the unfilled `template`, substituted `severity_word`, `arc_length_parallel`, and `arc_length_orthogonal` (legacy name for snapped PLS3 height). Prompts outside saved height coverage snap to the nearest saved slice and are counted below.

In [ ]:
from scripts import severity_inference

severity_csv, severity_rows, severity_diagnostics = severity_inference.run(config, force=FORCE)
display(severity_rows)
display(severity_diagnostics.groupby('coordinate_status').size().rename('rows').to_frame())
display(severity_diagnostics[['outside_saved_height_range', 'surface_extended', 'height_extrapolated']].sum().rename('rows').to_frame())
display(severity_diagnostics[['surface_projection_residual', 'slice_height_error']].describe())
print('Severity CSV:', severity_csv)

## 15. Inference-only severity wording prompts

The 20 fixed tasks in `scripts/corpora/severity_wording_prompts.py` each start with eight independently editable wording variants (160 prompts). Only wording and expressed distress vary; no severity ranking is assigned. These prompts never enter fitting or slice-grid construction.

Outputs are separate under `artifacts/<model name>/inference/severity_wording/`. The CSV contains `task`, the full `prompt`, `arc_length_parallel`, and `arc_length_orthogonal` (snapped PLS3 height). Valid activation caches are reused; coordinates are recomputed through the current saved surface. `FORCE` controls cache rebuilding. Prompts outside saved height coverage snap to the nearest saved slice and are reported below.

In [ ]:
from scripts import severity_wording_inference

wording_csv, wording_rows, wording_diagnostics = severity_wording_inference.run(config, force=FORCE)
display(wording_rows)
display(wording_diagnostics.groupby('coordinate_status').size().rename('rows').to_frame())
display(wording_diagnostics[['outside_saved_height_range', 'surface_extended', 'height_extrapolated']].sum().rename('rows').to_frame())
display(wording_diagnostics[['surface_projection_residual', 'slice_height_error']].describe())
print('Severity wording CSV:', wording_csv)